# Run Video QA Streamlit App (v3) on Colab (GPU) via Cloudflare Tunnel

**Before running:** set the runtime to GPU --> *Runtime > Change runtime type > Hardware accelerator > GPU (T4)*.

v3 flow: **Analyze** shows only Normal/Anomalous + Summary; details (people/weapon/location/category/actions) are asked on demand in the VQA section.

Run the cells top-to-bottom. The last cell prints a public `https://*.trycloudflare.com` URL.

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/sudeeprana8043-svg/Streamlit_project.git"
REPO_DIR = "/content/Streamlit_project"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Configure model files

`MODEL_DIR` stays pointed at the repo's `model/` folder (ships the temporal-matched legacy files). This cell copies the `models3` artifacts in (skipping colliding encoders), pulls `temporal_adapter.pt` from the old `/models` folder, and loads the summarization checkpoint `checkpoint-414` into the `checkpoint-140` folder the app expects.

In [ ]:
import os, shutil

from google.colab import drive
drive.mount("/content/drive")

MODEL_LOCAL  = os.path.abspath("model_instruct")  # Changed to use Instruct-based models
os.makedirs(MODEL_LOCAL, exist_ok=True)

DRIVE_MODELS = "/content/drive/MyDrive/models3"

# Files in models3 that COLLIDE with the temporal-matched repo files -> keep repo originals.
SKIP = {
    "le_weapon.pkl", "le_location.pkl", "le_people.pkl", "le_super.pkl",
    "model_config.pkl", "binary_model.pkl", "model_metadata.pkl",
}

copied, skipped = [], []
for fname in sorted(os.listdir(DRIVE_MODELS)):
    src = os.path.join(DRIVE_MODELS, fname)
    if not os.path.isfile(src):
        continue
    if fname in SKIP:
        skipped.append(fname)
        continue
    shutil.copy(src, os.path.join(MODEL_LOCAL, fname))
    copied.append(fname)
print(f"Copied {len(copied)} files from models3 -> model_instruct/")
print(f"Skipped (kept repo originals): {skipped}")

OLD_DRIVE_MODELS = "/content/drive/MyDrive/models"
for legacy in ["temporal_adapter.pt", "binary_model.pkl", "model_config.pkl"]:
    dst = os.path.join(MODEL_LOCAL, legacy)
    src = os.path.join(OLD_DRIVE_MODELS, legacy)
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied legacy {legacy} from {OLD_DRIVE_MODELS} -> model_instruct/")

SUMMARY_CKPT_DRIVE = "/content/drive/MyDrive/Project_VLM/ucf_qwen_v9_qformer/checkpoint-414"
SUMMARY_CKPT_LOCAL = os.path.join(MODEL_LOCAL, "checkpoint-140")
if not os.path.isdir(SUMMARY_CKPT_DRIVE):
    raise FileNotFoundError(f"Summary checkpoint not found in Drive: {SUMMARY_CKPT_DRIVE}")
shutil.copytree(SUMMARY_CKPT_DRIVE, SUMMARY_CKPT_LOCAL, dirs_exist_ok=True)
os.environ["SUMMARY_CHECKPOINT"] = SUMMARY_CKPT_LOCAL
os.environ["LORA_CHECKPOINT"] = SUMMARY_CKPT_LOCAL
print(f"Loaded checkpoint-414 weights into {SUMMARY_CKPT_LOCAL}")

os.environ["MODEL_DIR"] = MODEL_LOCAL
os.environ["BINARY_MODEL_DIR"] = MODEL_LOCAL

for f in ["multiclass_bundle.pkl", "simple_adapter.pt", "lora_model.pt", "input_dim.pkl"]:
    print(f"  {f}:", "ok" if os.path.exists(os.path.join(MODEL_LOCAL, f)) else "MISSING")

## 4. Download the Cloudflare tunnel binary

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

## 5. Launch Streamlit (v3) + public tunnel

In [ ]:
import subprocess, time, os, sys

PORT = 8501
APP = "streamlit_app_v3.py"  # v3 = analyze (status+summary) then on-demand VQA

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", APP,
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(f"Starting {APP}... (give it ~20s to boot and load models)")
time.sleep(20)
print("Recent Streamlit log:")
!tail -n 20 /content/streamlit.log

print("\n=== Public URL will appear below (look for *.trycloudflare.com) ===\n")
!./cloudflared tunnel --url http://localhost:$PORT